거의 모든 프로그램에는 데이터 파싱이나 결과물 출력과 같은 텍스트 프로세싱이 들어 있다. 

이번 장은 문자열 나누기, 검색, 빼기, 렉싱, 파싱 같은 텍스트 처리와 관련 있는 문제에 초점을 맞춘다. 

## 2.1 여러 구분자로 문자열 나누기

문제: 문자열을 필드로 나누고 싶지만 구분자(그리고 그 주변의 공백)가 문자열에 일관적이지 않다. 

In [1]:
import re
line = 'asdf fjdk; afed, fjek,asdf,      foo'

In [2]:
re.split(r'[;,\s]\s*',line)

['asdf', 'fjdk', 'afed', 'fjek', 'asdf', 'foo']

토론 re.split()함수는 분리 구문마다 여러 패턴을 명시할 수 있다는 점이 유리하다. 예를 들어 앞의 코드에서 본 대로 쉼표(,)나 세미콜론(;), 공백문자와 뒤이어 나오는 하나 이상의 공백문자 모두를 분리 구문으로 사용했다. 이 패턴이 나올 때마다 매칭된 부분 모두가 구분자가 된다. 결과는 str.split()와 마찬가지로 필드 리스트가 된다.  

## 2.2 문자열 처음이나 마지막에 텍스트 매칭

문제: 문자열의 처음이나 마지막에 파일 확장자, URL, 스킴(scheme)등 특정한 패턴이 포함되었는지 검사하고 싶다. 

해결

문자열의 처음이나 마지막에 패턴이 포함되었는지 확인한다.
```str.startswith(), str.endswith()```같은 메소드가 있다. 

In [3]:
filename = 'spam.txt'
filename.endswith('.txt')

True

In [4]:
filename.startswith('file:')

False

In [6]:
url = 'http://www.python.org'
url.startswith('http:')

True

여러개의 선택지를 검사해야 한다면 검사하고 싶은 값을 튜플에 담아 startswith()나 endswith()에 전달 한다. 

In [10]:
import os
filenames = os.listdir('.')
filenames.append('example.txt')

In [11]:
[name for name in filenames if name.endswith(('.ipynb','.txt'))]

['algorithm.ipynb', 'textforstring.ipynb', 'example.txt']

In [12]:
files = ['data.txt','script.py','example.csv','test.py','test.csv']

In [19]:
for i in files:
    if i.endswith(('.txt','.csv')):
        print(i)

data.txt
example.csv
test.csv


In [20]:
from urllib.request import urlopen

def read_data(name):
    if name.startswith(('http:','https:','ftp:')):
        return urlopen(name).read()
    else:
        with open(name) as f:
            return f.read()

In [22]:
choices = ['http:', 'ftp:']
url = 'http://www.python.org'
url.startswith(tuple(choices))

True

토론

슬라이스를 사용할 수도 있으나 가독성이 떨어진다. 정규표현식을 사용할 수도 있다. 

In [23]:
import re
url = 'http://www.python.org'
re.match('http:|https:|ftp:',url)

<re.Match object; span=(0, 5), match='http:'>

디렉터리에서 특정 파일이 있는지 확인하는데 좋다!

## 2.3 쉘 와일드카드 패턴으로 문자열 매칭

문제: Unix쉘에 사용하는 것과 동일한 와일드카드 패턴을 텍스트 매칭에 사용하고 싶다. (예:*.py, Dat[0-9]*.csv등)

In [26]:
from fnmatch import fnmatch, fnmatchcase
fnmatch('foo.txt','*.txt')

True

In [27]:
fnmatch('foo.txt','?oo.txt')

True

In [28]:
fnmatch('Dat45.csv','Dat[0-9]*')

True

In [30]:
names = ['Dat1.csv','Dat2.csv','config.ini','foo.py']
[name for name in names if fnmatch(name, 'Dat*.csv')]

['Dat1.csv', 'Dat2.csv']

윈도우에서 fnmatch는 대소문자 구별을 하지 않는다.!! 

In [34]:
fnmatch('foo.txt','*.TXT') #window는 True

False

In [36]:
fnmatchcase('foo.txt','*.TXT') #그럴때 사용가능한 함수

False

이 함수의 중요한 기능은 파일이름이 아닌 데이터 프로세싱에도 사용할 수 있다는 사실이다. 예를 들어 다음과 같은 주소 리스트가 있다고 가정해보자

In [37]:
addresses = [
    '5412 N CLARK ST',
    '1060 W ADDISON ST',
    '1039 W GRANVILLE AVE',
    '2122 N CLARK ST',
    '4802 N BROADWAY'
]

In [38]:
[addr for addr in addresses if fnmatchcase(addr, '* ST')]

['5412 N CLARK ST', '1060 W ADDISON ST', '2122 N CLARK ST']

토론 

fnmatch가 수행하는 매칭은 간단한 문자열 메소드의 기능과 정규 표현식의 중간쯤 위치하고 있다. 데이터 프로세싱을 할 때 간단한 와일드 카드를 사용할 생각이라면 이 함수를 사용하는 것은 괜찮은 선택이다. 

파일 이름을 찾는 코드를 실제로 작성해야 한다면 이 함수 대신 glob모듈을 사용하도록 한다. 

## 2.4 텍스트 패턴 매칭과 검색

문제: 특정 패턴에 대한 텍스트 매칭이나 검색을 하고 싶다.

매칭하려는 텍스트가 간단하다면 기본적인 메소드 만으로도 충분하다. 

```
str.find()
str.startswith()
str.endswith()
```

In [41]:
text = 'yeah, but no, but yeah, but no, but yeah'

In [42]:
text == 'yeah'

False

In [43]:
text.startswith('yeah')

True

In [44]:
text.endswith('no')

False

In [45]:
text.find('no')

10

In [46]:
text1 = '11/27/2012'
text2 = 'Nov 27, 2012'

In [47]:
import re
if re.match(r'\d+/\d+/\d',text1):
    print('yes')
else:
    print('no')

yes


In [48]:
if re.match(r'\d+/\d+/\d',text2):
    print('yes')
else:
    print('no')

no


동일한패턴으로 매칭을 많이 수행할 예정이라면 정규 표현식을 미리 컴파일해서 패턴 객체로 만들어 놓는 것이 좋다. 

In [50]:
datapat = re.compile(r'\d+/\d+/\d')
if datapat.match(text1):
    print('yes')
else:
    print('no')

yes


In [51]:
if datapat.match(text2):
    print('yes')
else:
    print('no')

no


match()는 항상 문자열 처음에서 찾기를 시도한다. 텍스트 전체에 걸쳐 패턴을 찾으려면 findall()메소드를 사용한다. 

In [53]:
text = 'Today is 11/27/2012. PyCon starts 3/13/2013.'
datapat.findall(text)

['11/27/2', '3/13/2']

정규표현식을 정의할 때 괄호를 사용해 캡쳐 그룹을 만드는 것이 일반적이다.

In [67]:
datapet = re.compile(r'(\d+)/(\d+)/(\d+)')

캡쳐 그룹을 사용하면 매칭된 텍스트에 작업할 때 개별적으로 추출할 수 있다. 

In [68]:
m = datapet.match('11/27/2012')
m

<re.Match object; span=(0, 10), match='11/27/2012'>

In [69]:
m.group(0)

'11/27/2012'

In [71]:
m.group(1)

'11'

In [72]:
m.group(2)

'27'

In [74]:
m.groups()

('11', '27', '2012')

In [75]:
month, day, year = m.groups()

In [76]:
text

'Today is 11/27/2012. PyCon starts 3/13/2013.'

In [77]:
datapet.findall(text)

[('11', '27', '2012'), ('3', '13', '2013')]

In [81]:
for month, day, year in datapet.findall(text):
    print(month, day, year)
    print('{}-{}-{}'.format(year, month, day))

11 27 2012
2012-11-27
3 13 2013
2013-3-13


findall()메소드는 텍스트를 검색하고 모든 매칭을 찾아 리스트로 반환한다. 

한번에 결과를 얻지 않고 텍스트를 순환하며 찾으려면 finditer()를 사용한다.

In [82]:
for m in datapet.finditer(text):
    print(m.groups())

('11', '27', '2012')
('3', '13', '2013')


토론

정규 표현식의 모든 내용을 다루진 않았지만 문자열 검색과 매칭을 위해 re모듈을 이용하는 기본적인 방법을 알아보았다. 

핵심이 되는 기능은 ```re.compile()```을 사용해 패턴을 컴파일하고 그것을 ```match(), findall(), finditer()```등에 사용한다는 점이다. 

패턴을 명시할때 ```r'(\d+)/(\d+)/(\d+)'```와 같이 로우 문자열(raw string)을 그대로 쓰는 것이 일반적이다. 로우 문자열을 사용하지 않으면 ```'(\\d+)/(\\d+)/(\\d+)'```와 같이 백슬래시를 두 번 사용해야 하는 불편함이 따른다. 

match()메소드는 문자열의 처음만 확인한다는 점을 주의하자

In [86]:
m = datapet.match('11/27/2012abcdef')
m

<re.Match object; span=(0, 10), match='11/27/2012'>

In [87]:
m.group()

'11/27/2012'

정확한 매칭을 위해서 패턴의 문자열에 마지막을 나타내는 $부호를 사용한다. 

In [89]:
datepat = re.compile(r'(\d+)/(\d+)/(\d+)$')
datepat.match('11/27/2012abcdef')

In [90]:
datepat.match('11/27/2012')

<re.Match object; span=(0, 10), match='11/27/2012'>

## 2.5 텍스트 검색과 치환

문제: 문자열에서 텍스트 패턴을 검색하고 치환하고 싶다. 

In [91]:
text = 'yeah, but no, but yeah, but no, but yeah'

In [92]:
text.replace('yeah','yep')

'yep, but no, but yep, but no, but yep'

조금더 복잡한 패턴을 사용하려면 re모듈의 sub()함수/메소드를 사용한다. 예를 들어 11/27/2012형식의 날짜를 "2012-11-27"로 바꿔야 할 상황을 가정해 보자

In [93]:
text = "Today is 11/27/2012. PyCon starts 3/13/2013"
import re
re.sub(r'(\d+)/(\d+)/(\d+)', r'\3-\1-\2', text)

'Today is 2012-11-27. PyCon starts 2013-3-13'

sub()에 전달한 첫 번째 인자는 매칭을 위한 패턴이고 두 번째 인자는 치환을 위한 패턴이다. 숫자 앞에 백슬래시가 붙어 있는 \3과 같은 표현은 패턴의 캡처 그룹을 참조한다.

동일한 패턴을 사용한 치환을 계속한다면 성능 향상을 위해 컴파일링을 고려해 보는 것도 좋다. 

In [94]:
import re
datepat = re.compile(r'(\d+)/(\d+)/(\d+)')
datepat.sub(r'\3-\1-\2', text)

'Today is 2012-11-27. PyCon starts 2013-3-13'

더 복잡한 치환을 위해서 콜백 함수를 명시할 수도 있다.

In [95]:
from calendar import month_abbr
def change_date(m):
    mon_name = month_abbr[int(m.group(1))]
    return '{} {} {}'.format(m.group(2), mon_name, m.group(3))

In [96]:
datepat.sub(change_date, text)

'Today is 27 Nov 2012. PyCon starts 13 Mar 2013'

인자가 되는 치환 콜백은 match()나 find()에서 반환한 매치 객체를 사용한다. 매치에서 특정 부분을 추출하려면 .group()메소드를 사용한다. 이 함수는 치환된 텍스트를 반환해야 한다. 

만약 치환된 텍스트를 받기 전에 치환이 몇 번 발생했는지 알고 싶다면 re.subn()을 사용한다.

In [97]:
newtext, n = datepat.subn(r'\3-\1-\2', text)
newtext

'Today is 2012-11-27. PyCon starts 2013-3-13'

토론: 앞서 살펴본 sub()메소드에 정규 표현식 검색과 치환 이외에 어려운 것은 없다. 가장 어려운 것은 정규 표현식 패턴을 만드는 것인데 이는 스스로 연습해야 한다. 

## 2.6 대소문자를 구별하지 않는 검색과 치환

문제: 텍스트를 검색하고 치환할 때 대소문자를 구별하지 않고 싶다.

해결: 텍스트 관련 작업을 할 때 대소문자를 구별하지 않기 위해서는 re모듈을 사용해야 하고 re.IGNORECASE 플래그를 지정해야 한다.

In [98]:
text = 'UPPER PYTHON, lower python, Mixed Python'
re.findall('python', text, flags=re.IGNORECASE)

['PYTHON', 'python', 'Python']

In [99]:
re.sub('python', 'snake', text, flags=re.IGNORECASE)

'UPPER snake, lower snake, Mixed snake'

앞에 나온 예제를 보면 치환된 텍스트의 대소문자가 원본의 대소문자와 일치하지 않는다. 이 부분을 고치려면 다음과 같이 함수를 하나 만들어 제공한다. 

In [100]:
def matchcase(word):
    def replace(m):
        text = m.group()
        if text.isupper():
            return word.upper()
        elif text.islower():
            return word.lower()
        elif text[0].isupper():
            return word
    return replace

In [101]:
re.sub('python', matchcase('snake'), text, flags=re.IGNORECASE)

'UPPER SNAKE, lower snake, Mixed snake'

토론: 대개의 경우 re.IGNORECASE를 사용한 것만으로 대소문자를 무시한 텍스트 작업에 무리가 없다. 하지만 유니코드(UNICODE)가 포함된 작업을 하기에는 부족할 수도 있다. 레시피 2.10에 이에 대한 해결책이 나와 있다. 

## 2.7 가장 짧은 매칭을 위한 정규 표현식

문제: 정규 표현식을 사용한 텍스트 매칭을 하고 싶지만 텍스트에서 가장 긴 부분을 찾아낸다. 만약 가장 짧은 부분을 찾아내고 싶다면 어떻게 해야 할까?

해결: 이런 문제는 문장 구분자에 둘러싸여 있는 텍스트를 찾을 때 종종 발생한다. (예를 들어 인용문 등)

In [102]:
str_pat = re.compile(r'\"(.*)\"')
text1 = 'Computer says "no."'
str_pat.findall(text1)

['no.']

In [103]:
text2 = 'Computer says "no. " Phone says "yes. "'
str_pat.findall(text2)

['no. " Phone says "yes. ']